In [17]:
import pandas as pd
import matplotlib
matplotlib.use('TkAgg')


In [18]:
import plotly
plotly.version

'6.5.2'

In [19]:
!ls

PCB153-vs-CTR-RBP-DEGS.txt  rbp-degs-graph.ipynb      VOLCANO_PLOT.png
RBPbench		    tmp_rbp-degs-graph.ipynb


In [20]:
degs='/home/maria/Desktop/progetto_inquinanti/risultati_inquinanti/PCB153-vs-CTR/DEGS/PCB153-vs-CTR-DEGs.tsv'
degs=pd.read_csv(degs,sep='\t')
len(degs.Gene.to_list())

1882

In [21]:
rbp='/home/maria/Desktop/progetto_inquinanti/rbp_list/rbp_list.txt'
rbp_names=pd.read_csv(rbp,sep ='\t', names=['RBP'])
RBP_names=rbp_names.RBP.tolist()
len(RBP_names)

256

In [22]:
!ls /home/maria/Desktop/progetto_inquinanti/risultati_inquinanti/PCB153-vs-CTR/DEGS


all_SH-SY5Y-PCB_CTRL-WT_PCB153-5uM_Tables_DEGs_gene_minimum_count_list.tsv
PCB153-vs-CTR-DEGs.tsv


In [23]:
all_degs='/home/maria/Desktop/progetto_inquinanti/risultati_inquinanti/PCB153-vs-CTR/DEGS/all_SH-SY5Y-PCB_CTRL-WT_PCB153-5uM_Tables_DEGs_gene_minimum_count_list.tsv'
df=pd.read_csv(all_degs,sep ='\t')
df

,Unnamed: 0,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
0,A1BG-AS1,26.533414,0.493432,0.670206,0.736240,0.461585,0.752584
1,A2M-AS1,18.547891,-0.928632,0.815633,-1.138542,0.254894,0.559455
2,A3GALT2,2.541543,-0.034715,0.873595,-0.039738,0.968302,0.990795
3,AAAS,2135.579846,-0.017872,0.071985,-0.248267,0.803928,0.933184
4,AACS,396.360816,-0.174626,0.202570,-0.862055,0.388657,0.693960
...,...,...,...,...,...,...,...
15809,ZYG11A,128.719971,0.359989,0.230404,1.562424,0.118188,0.358669
15810,ZYG11B,1201.447882,-0.012876,0.092837,-0.138694,0.889692,0.964872
15811,ZYX,442.003008,-0.118825,0.068426,-1.736543,0.082468,0.288401
15812,ZZEF1,2196.583064,-0.118466,0.046646,-2.539670,0.011096,0.078451


In [24]:
rbp_degs=pd.read_csv('PCB153-vs-CTR-RBP-DEGS.txt')
genes_to_label=rbp_degs.Gene.to_list()
len(genes_to_label)

32

In [25]:
df

,Unnamed: 0,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
0,A1BG-AS1,26.533414,0.493432,0.670206,0.736240,0.461585,0.752584
1,A2M-AS1,18.547891,-0.928632,0.815633,-1.138542,0.254894,0.559455
2,A3GALT2,2.541543,-0.034715,0.873595,-0.039738,0.968302,0.990795
3,AAAS,2135.579846,-0.017872,0.071985,-0.248267,0.803928,0.933184
4,AACS,396.360816,-0.174626,0.202570,-0.862055,0.388657,0.693960
...,...,...,...,...,...,...,...
15809,ZYG11A,128.719971,0.359989,0.230404,1.562424,0.118188,0.358669
15810,ZYG11B,1201.447882,-0.012876,0.092837,-0.138694,0.889692,0.964872
15811,ZYX,442.003008,-0.118825,0.068426,-1.736543,0.082468,0.288401
15812,ZZEF1,2196.583064,-0.118466,0.046646,-2.539670,0.011096,0.078451


In [33]:
filtered_df = df[(df['log2FoldChange'] >= 0.5) | (df['log2FoldChange'] <= -0.5)]
filtered_df['gene'] = filtered_df['Unnamed: 0']
filtered_df['significance'] = np.where(filtered_df['padj'] < 0.05, 'Significant', 'Non-Significant')
rbp_significant_fold_change = filtered_df[(filtered_df['gene'].isin(genes_to_label)) & (filtered_df['significance']=='Significant')]
sig_rbp=rbp_significant_fold_change.gene.to_list()

/tmp/ipykernel_33742/873963063.py:2: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_33742/873963063.py:3: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [36]:
import dash
from dash import dcc, html
import plotly.graph_objects as go
import pandas as pd
import numpy as np

# Data prep
df['gene'] = df['Unnamed: 0']
#df['pvalue'] = df['pvalue'].replace(0, 1e-300)
df['log10_padj'] = -np.log10(df['padj'])
df['significance'] = np.where(df['padj'] < 0.05, 'Significant', 'Non-Significant')

# Split into categories
significant = df[(df['significance']=='Significant') & (~df['gene'].isin(genes_to_label))]
non_significant = df[df['significance']=='Non-Significant']
rbp_significant = df[(df['gene'].isin(genes_to_label)) & (df['significance']=='Significant')]

# Create figure
fig = go.Figure()

# Non-significant DEGs (gray)
fig.add_trace(go.Scatter(
    x=non_significant['log2FoldChange'],
    y=non_significant['log10_padj'],
    mode='markers',
    marker=dict(color='lightgray', size=7),
    name='Non-Significant DEGs',
    text=non_significant['gene'],
    hovertemplate='<b>%{text}</b><br>log2FC: %{x}<br>-log10(p): %{y}<extra></extra>'
))

# Significant DEGs (green)
fig.add_trace(go.Scatter(
    x=significant['log2FoldChange'],
    y=significant['log10_padj'],
    mode='markers',
    marker=dict(color='blue', size=10),
    name='Significant DEGs',
    text=significant['gene'],
    hovertemplate='<b>%{text}</b><br>log2FC: %{x}<br>-log10(p): %{y}<extra></extra>'
))

# Significant RBP DEGs (red)
fig.add_trace(go.Scatter(
    x=rbp_significant['log2FoldChange'],
    y=rbp_significant['log10_padj'],
    mode='markers',
    marker=dict(color='#FF6D00', size=10),
    name='Significant RBP DEGs',
    text=rbp_significant['gene'],
    hovertemplate='<b>%{text}</b><br>log2FC: %{x}<br>-log10(p): %{y}<extra></extra>'
))

# Add annotations for RBP genes
annotations = []
offset_y = 0
for i, row in rbp_significant.iterrows():
    annotations.append(dict(
        x=row['log2FoldChange'],
        y=row['log10_padj'],
        xref='x',
        yref='y',
        text=row['gene'],
        showarrow=True,
        arrowhead=2,
        ax=20,
        ay=-20 + offset_y,
        bgcolor='white',
        bordercolor='black'
    ))
    offset_y = -offset_y if offset_y == 0 else 0

# ------------------ Significance threshold line ------------------ #
threshold = -np.log10(0.05)
fig.add_shape(
    type='line',
    x0=-5.5, x1=5.5,
    y0=threshold, y1=threshold,
    line=dict(color='black', width=2, dash='dash')
)


fig.update_layout(
    title='Volcano Plot: PCB153 vs CTR',
    xaxis_title='log<sub>2</sub> Fold Change',                  # 2 as subscript
    yaxis_title='−log<sub>10</sub>(adjusted p-value)',         # 10 as subscript + proper minus
    xaxis=dict(range=[-5.5,5.5]),
    yaxis=dict(range=[0,25]),
    template='plotly_white',
    annotations=annotations,
    dragmode='pan',
    width=1400,
    height=1000,
    legend=dict(
        x=0.85,
        y=1,
        xanchor='left',
        yanchor='top',
        font=dict(size=22),
        bgcolor='rgba(255,255,255,0.85)',
    )
)

# ------------------ Dash App ------------------ #
app = dash.Dash(__name__)

app.layout = html.Div([
    dcc.Location(id='url', refresh=False),
    html.Div(id='page-content')
])
threshold = -np.log10(0.05)

fig.update_yaxes(
    range=[0, 25],
    tickvals=[0, 5, 10, 15, 20, threshold],  # add threshold to ticks
    ticktext=['0', '5', '10', '15', '20', 'padj=0.05']
)
# Multi-page callback
@app.callback(
    dash.dependencies.Output('page-content', 'children'),
    [dash.dependencies.Input('url', 'pathname')]
)
def display_page(pathname):
    if pathname == '/volcano':
        # Fully interactive volcano plot page with download button
        return html.Div([
            html.H1("Interactive Volcano Plot"),
            dcc.Graph(
                id='volcano-plot',
                figure=fig,
                config={
                    'editable': True,
                    'displayModeBar': True,
                    'displaylogo': False,
                    'toImageButtonOptions': {
                        'format': 'png',  # png, jpeg, svg
                        'scale': 3        # ~300 DPI
                    }
                },
                style={'height': '90vh'}
            ),
            html.P("Move labels as needed, then click the camera icon to download a 300 DPI PNG.")
        ])
    # Home page
    return html.Div([
        html.H1("Home Page"),
        html.A("Open Volcano Plot in New Tab", href="/volcano", target="_blank")
    ])

if __name__ == '__main__':
    app.run(debug=True)


/home/maria/anaconda3/lib/python3.9/site-packages/pandas/core/arraylike.py:399: RuntimeWarning:

divide by zero encountered in log10

